# Player Level & Game Day Progression Analysis

**Purpose:** Understand how quickly players progress through levels and game days across install cohorts, and how retention curves compare over time.

**Data range:** April 2024 – April 2026 install cohorts (~59M player-day observations)

---

## Key Findings

- **Level progression is rapid in the first few days:** Players reach an average of ~4 levels on install day (D0), ~6 by D1, ~7 by D2, and ~8 by D3.
- **Recent cohorts show slightly lower early engagement:** Average max level on D0 dropped from ~4.0 (Apr 2024) to ~3.5 (Apr 2026) — roughly an 11% decrease.
- **D1 retention has declined slightly:** ~55% of the Apr 2024 cohort returned on D1 vs ~51% for Apr 2026, suggesting a softening in early-day engagement over time.
- **Wide player distribution:** P90 players progress ~3–4× faster than P10, and this spread widens significantly beyond D7.
- **Game day and level are tightly coupled:** Game day progression closely mirrors level progression (~2 game days on D0, ~3.5 on D1), indicating most sessions drive both metrics in tandem.

In [4]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np

bqc = BigQueryConnector()

## Get data

### Player level and game day

In [13]:
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/playerlevel.sql'
parameters = {
    'start_date': '2024-04-01',
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 248.14 GB when run.
Estimated query cost: $1.67


In [ ]:
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache
refresh_data = False
if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [15]:
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,acquisition_type
0,D1AA8C2528335FE9,2025-08-03,2025-04-17,2025-04-13,2025-04-01,108,30,27,UA
1,2F42BA9C95563FBA,2025-07-23,2025-06-02,2025-06-01,2025-06-01,51,29,25,Non-Attributed
2,8C9C5AD42D250FFB,2026-02-14,2025-04-21,2025-04-20,2025-04-01,299,44,46,Non-Attributed
3,3964FBF07ECD8847,2026-01-04,2025-12-16,2025-12-14,2025-12-01,19,12,8,Non-Attributed
4,E14BEF008F9905D2,2026-03-11,2025-12-01,2025-11-30,2025-12-01,100,41,43,Non-Attributed
...,...,...,...,...,...,...,...,...,...
60284476,3497F3B65BF8EEF,2024-06-20,2024-04-07,2024-04-07,2024-04-01,74,41,41,CPE
60284477,A7FAB4F140B425C5,2026-02-21,2025-03-23,2025-03-23,2025-03-01,335,78,91,Non-Attributed
60284478,16B363905A4B1564,2024-08-03,2024-07-12,2024-07-07,2024-07-01,22,13,9,CPE
60284479,C6D59545F3AEC26F,2025-05-15,2024-12-31,2024-12-29,2024-12-01,135,49,53,CPE


## Process data

In [16]:
dt_mode = 'install_dt_month'

data['install_dt'] = data[dt_mode]

data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in data['acquisition_type']]

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
data = data[data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(data['install_dt'])).dt.days - min_days_since_install]
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,acquisition_type,CPE_flag
0,D1AA8C2528335FE9,2025-08-03,2025-04-01,2025-04-13,2025-04-01,108,30,27,UA,N
1,2F42BA9C95563FBA,2025-07-23,2025-06-01,2025-06-01,2025-06-01,51,29,25,Non-Attributed,N
2,8C9C5AD42D250FFB,2026-02-14,2025-04-01,2025-04-20,2025-04-01,299,44,46,Non-Attributed,N
3,3964FBF07ECD8847,2026-01-04,2025-12-01,2025-12-14,2025-12-01,19,12,8,Non-Attributed,N
4,E14BEF008F9905D2,2026-03-11,2025-12-01,2025-11-30,2025-12-01,100,41,43,Non-Attributed,N
...,...,...,...,...,...,...,...,...,...,...
60284476,3497F3B65BF8EEF,2024-06-20,2024-04-01,2024-04-07,2024-04-01,74,41,41,CPE,Y
60284477,A7FAB4F140B425C5,2026-02-21,2025-03-01,2025-03-23,2025-03-01,335,78,91,Non-Attributed,N
60284478,16B363905A4B1564,2024-08-03,2024-07-01,2024-07-07,2024-07-01,22,13,9,CPE,Y
60284479,C6D59545F3AEC26F,2025-05-15,2024-12-01,2024-12-29,2024-12-01,135,49,53,CPE,Y


In [17]:
# Cache the filtered dataset to avoid reprocessing on subsequent runs
data.to_pickle('./data/playprogression_processed.pkl')

In [8]:
# Load the cohort-filtered processed dataset
data = pd.read_pickle('./data/playprogression_processed.pkl') 

## Player Level reached at day x

For each install cohort, tracks the **weighted average max level** reached as a function of days since install. Weighted average is used to account for varying player counts across level buckets. The percentage-change chart below highlights where the steepest level gains occur in the early-day window.

In [18]:
def compute_weighted_progression(data, measure_col, dimension_cols=['install_dt', 'days_since_install'], min_bucket_size=50):
    """
    Compute weighted average progression metric for a given measure across dimensions.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Source data containing user_id, the measure column, and dimension columns
    measure_col : str
        Column name to compute weighted average for (e.g., 'max_level', 'max_gameday')
    dimension_cols : list
        Dimensions to group by (default: ['install_dt', 'days_since_install'])
    min_bucket_size : int
        Minimum users per bucket to include (default: 50)
    
    Returns:
    --------
    pd.DataFrame
        Aggregated data with weighted average and cohort user counts
    """
    
    # Step 1: Count unique users per dimension + measure bucket
    agg = data.groupby(dimension_cols + [measure_col]).agg(
        unique_users=('user_id', 'nunique')
    ).reset_index()
    
    # Step 2: Total unique users per dimension combination
    dimension_total_users = data.groupby(dimension_cols).agg(
        total_unique_users=('user_id', 'nunique')
    ).reset_index()
    
    # Step 3: Merge and compute percentage share
    agg = agg.merge(dimension_total_users, on=dimension_cols)
    agg['percentage_of_users'] = agg['unique_users'] / agg['total_unique_users']
    
    # Step 4: Compute weighted average
    weighted_avg = agg.groupby(dimension_cols, group_keys=False).apply(
        lambda x: (x[measure_col] * x['unique_users']).sum() / x['unique_users'].sum(),
        include_groups=False
    ).reset_index()
    
    weighted_avg.columns = dimension_cols + [f'weighted_avg_{measure_col}']
    agg = agg.merge(weighted_avg, on=dimension_cols)
    
    # Step 5: Drop small buckets
    agg = agg[agg['unique_users'] >= min_bucket_size]
    
    # Step 6: Collapse to one row per dimension combination
    agg = agg.groupby(dimension_cols).agg(
        cohort_users=('unique_users', 'sum'),
        **{f'weighted_avg_{measure_col}': ('weighted_avg_' + measure_col, 'first')}
    ).reset_index()
    
    return agg

In [19]:
player_level_agg = compute_weighted_progression(data, measure_col='max_level', dimension_cols=['install_dt', 'days_since_install','CPE_flag'], min_bucket_size=50)
player_level_agg['combined_dimension'] = player_level_agg['install_dt'].astype(str) + ' | ' + player_level_agg['CPE_flag']
player_level_agg

,install_dt,days_since_install,CPE_flag,cohort_users,weighted_avg_max_level,combined_dimension
0,2024-04-01,0,N,61161,3.828078,2024-04-01 | N
1,2024-04-01,0,Y,61343,4.129031,2024-04-01 | Y
2,2024-04-01,1,N,32729,5.862795,2024-04-01 | N
3,2024-04-01,1,Y,34768,5.904672,2024-04-01 | Y
4,2024-04-01,2,N,25733,6.945698,2024-04-01 | N
...,...,...,...,...,...,...
15447,2026-04-01,18,Y,2927,16.906386,2026-04-01 | Y
15448,2026-04-01,19,N,1853,17.428889,2026-04-01 | N
15449,2026-04-01,19,Y,2845,17.281864,2026-04-01 | Y
15450,2026-04-01,20,N,1794,17.763547,2026-04-01 | N


In [20]:
# hide-output
fig = px.line(player_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='combined_dimension',
              title='Player level daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [21]:
def weighted_quantiles(group, quantiles=[0.1, 0.5, 0.9], measure_col='max_level'):
    """Return P10 / P50 / P90 of measure_col, using user counts as weights.

    Sorts by the measure, accumulates weights, then uses searchsorted to find
    the value at each quantile threshold — equivalent to a weighted percentile.
    """
    levels = group[measure_col].values
    weights = group['users'].values
    sorted_idx = np.argsort(levels)
    levels, weights = levels[sorted_idx], weights[sorted_idx]
    cum_weights = np.cumsum(weights)
    total = cum_weights[-1]
    result = {}
    for q in quantiles:
        idx = np.searchsorted(cum_weights, q * total)
        result[f'p{int(q * 100)}'] = levels[min(idx, len(levels) - 1)]
    return pd.Series(result)

In [ ]:
# hide-output
level_dist = data.groupby(['days_since_install', 'max_level','CPE_flag']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts = level_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_level', include_groups=False).reset_index()

fig = px.line(
    level_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_level'),
    x='days_since_install',
    y='max_level',
    color='percentile',
    title='Player level distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

In [23]:
player_level_agg2 = player_level_agg[['install_dt', 'days_since_install','weighted_avg_max_level']].drop_duplicates()

# Day-over-day % change in weighted avg level within each install cohort.
# Day 0 is filled as 1.0 (100%) since there is no prior day to compare against.
player_level_agg2['pct_change_weighted_avg_max_level'] = player_level_agg2.groupby('install_dt')['weighted_avg_max_level'].pct_change()
player_level_agg2.pct_change_weighted_avg_max_level = player_level_agg2.pct_change_weighted_avg_max_level.fillna(1)
player_level_agg2

,install_dt,days_since_install,weighted_avg_max_level,pct_change_weighted_avg_max_level
0,2024-04-01,0,3.828078,1.000000
1,2024-04-01,0,4.129031,0.078617
2,2024-04-01,1,5.862795,0.419896
3,2024-04-01,1,5.904672,0.007143
4,2024-04-01,2,6.945698,0.176306
...,...,...,...,...
15447,2026-04-01,18,16.906386,0.013140
15448,2026-04-01,19,17.428889,0.030906
15449,2026-04-01,19,17.281864,-0.008436
15450,2026-04-01,20,17.763547,0.027872


In [24]:
# hide-output
fig = px.line(player_level_agg2.where(player_level_agg2.days_since_install<=30), 
              x='days_since_install', 
              y='pct_change_weighted_avg_max_level',
              color='install_dt',
              title='Player level daily progression by cohort (pct change)',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True})


fig.show()

## Game day reached at day x

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [25]:
# Same weighted-average approach as player level, applied to max_gameday

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

,install_dt,days_since_install,cohort_users,weighted_avg_max_gameday
0,2024-04-01,0,122523,2.250436
1,2024-04-01,1,67501,3.581366
2,2024-04-01,2,53107,4.357179
3,2024-04-01,3,47339,4.956120
4,2024-04-01,4,43467,5.473805
...,...,...,...,...
8661,2026-04-01,16,5692,12.690193
8662,2026-04-01,17,5520,13.150209
8663,2026-04-01,18,5311,13.532577
8664,2026-04-01,19,5068,13.972798


In [26]:
# hide-output
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [27]:
# hide-output
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

## % of cohort users active by days since install

Retention curve: the share of a cohort's total users who were active on each calendar day since install. Plotted on a **log scale** so that differences between cohorts remain visible at longer horizons where absolute percentages are very small. Apr 2024 D1 retention was ~55%; Apr 2026 has declined to ~51%.

In [28]:
# Count unique users active on each day since install, per cohort
users_agg = pd.DataFrame()
users_agg = data.groupby(['install_dt', 'days_since_install']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Total unique users ever observed in each cohort (denominator for retention %)
users_by_cohort = pd.DataFrame()
users_by_cohort = data.groupby('install_dt').agg(
    total_cohort_users = ('user_id', 'nunique')
).reset_index()

users_agg = users_agg.merge(users_by_cohort, on='install_dt')

# Retention rate: share of cohort still active on a given day
users_agg['pct_cohort_users_active'] = users_agg['unique_users'] / users_agg['total_cohort_users']

users_agg

,install_dt,days_since_install,unique_users,total_cohort_users,pct_cohort_users_active
0,2024-04-01,0,122662,122753,0.999259
1,2024-04-01,1,67682,122753,0.551367
2,2024-04-01,2,53315,122753,0.434327
3,2024-04-01,3,47561,122753,0.387453
4,2024-04-01,4,43691,122753,0.355926
...,...,...,...,...,...
9634,2026-04-01,16,5894,30917,0.190639
9635,2026-04-01,17,5732,30917,0.185400
9636,2026-04-01,18,5556,30917,0.179707
9637,2026-04-01,19,5404,30917,0.174791


In [29]:
# hide-output

# apply a log transformation to the y axis to better visualize the differences between cohorts, especially in the later days since install where the percentage of active users is very low
fig = px.line(users_agg, 
              x='days_since_install', 
              y='pct_cohort_users_active',
              color='install_dt',
              title='% of cohort users active by days since install (log scale)',
              width=1200,
              height=600,
              hover_data={'pct_cohort_users_active': ':.2%', 'unique_users': True, 'total_cohort_users': True},
              log_y=True)


fig.show()

# Things to do next
- Is the churn increasing for P90 players when they reach the 150 GD mark?
- Is the slowdown on pace seen on around level 50 on newest cohort caused by CPE mix? what does it look like with only organics?